In [ ]:
import pandas as pd
import numpy as np
import sys
import os
_project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..', '..'))
if _project_root not in sys.path:
    sys.path.insert(0, _project_root)

from nimbus_testbench.stress_testing.scenario_engine.credit_risk import stress_test, StressTestResult
from nimbus_testbench.stress_testing.scenario_engine.credit_risk import (
    CreditRiskConfig, AccountingConfig, BaselConfig,
    ModelConfig, DataConfig, ValidationConfig,
)
from nimbus_testbench.stress_testing.scenario_engine.credit_risk import TTCToPITAdapter, DownturnLGDAdapter, StressedEADAdapter, PrecomputedModel, IFRS9StagingProcessor, TransitionMatrixModel, CreditCycleModel, ECLCalculator, BaselCapitalCalculator, ScenarioProcessor,   project_portfolio_forward

In [ ]:
portfolio     = pd.read_csv(r'D:\nimbus_testbench 6\nimbus_testbench\nimbus_testbench\stress_testing\scenario_engine\credit_risk\portfolio.csv', low_memory=False)
training_data = pd.read_csv(r'D:\nimbus_testbench 6\nimbus_testbench\nimbus_testbench\stress_testing\scenario_engine\credit_risk\training_data.csv', low_memory=False)
macro_history = pd.read_csv(r'D:\nimbus_testbench 6\nimbus_testbench\nimbus_testbench\stress_testing\scenario_engine\credit_risk\macro_history.csv')

baseline_raw  = pd.read_csv(r'D:\nimbus_testbench 6\nimbus_testbench\nimbus_testbench\stress_testing\scenario_engine\credit_risk\Table_2A_Supervisory_Baseline_Domestic.csv')
severe_raw    = pd.read_csv(r'D:\nimbus_testbench 6\nimbus_testbench\nimbus_testbench\stress_testing\scenario_engine\credit_risk\Table_3A_Supervisory_Severely_Adverse_Domestic.csv')

print(f'Portfolio:     {len(portfolio):,} loans | ${portfolio["CurrentBalance"].sum():,.0f} total')
print(f'Training data: {len(training_data):,} rows | {training_data["LoanID"].nunique():,} loans × {training_data["Time"].nunique()} quarters')
print(f'Macro history: {len(macro_history)} quarters | {macro_history["Date"].iloc[0]} to {macro_history["Date"].iloc[-1]}')
print(f'FRB baseline:  {len(baseline_raw)} quarters | {baseline_raw["Date"].iloc[0]} to {baseline_raw["Date"].iloc[-1]}')
print(f'FRB severe:    {len(severe_raw)} quarters')
print()
print('Portfolio Stage distribution:', portfolio['Stage'].value_counts().sort_index().to_dict())
print('Training data required cols present:', all(c in training_data.columns for c in ['LoanID','Time','Stage','Default','EAD','LGD_Observed','GDP_Growth','UNRATE']))

In [ ]:
accounting_cfg = AccountingConfig(
    framework='IFRS9',
    sicr_pd_multiplier=3.0,
    cure_pd_multiplier=2.0,
    cure_probation_quarters=2,
    cure_dpd_max=0,


    discount_rate_column='EffectiveInterestRate',
    compounding_frequency='quarterly',

    sicr_absolute_threshold_by_product={
        'Mortgage':    None,
        'Auto':        None,
        'Credit_Card': None,
        'Personal':    None,
    },

    sicr_qualitative_triggers=['ForbearanceFlag', 'WatchlistFlag'],
    scenario_weights_sum_tolerance=0.001,
)

basel_cfg = BaselConfig(
    irb_approach='Advanced',
    default_maturity_years=2.5,
    maturity_floor=1.0,
    maturity_cap=5.0,
    use_regulatory_rho_formula=True,
    lgd_floor_senior_secured=0.10,
    lgd_floor_senior_unsecured=0.20,
    lgd_floor_subordinated=0.35,
    gsib_surcharge=0.0,
    output_floor_enabled=True,
    sme_size_adjustment_enabled=False,
)

model_cfg = ModelConfig(
    pd_pit_or_ttc='PIT',
    vasicek_rho_initial=0.15,
    vasicek_rho_search_bounds=(0.05, 0.40),
    z_score_col='Z_Score',
    bridge_equation_vars=['GDP_Growth', 'UNRATE'],
    tm_min_obs_threshold=30,
    tm_credibility_k=50,
)

data_cfg = DataConfig(
    id_column='LoanID',
    time_column='Time',
    balance_column='CurrentBalance',
    pd_column='PD',
    pd_origination_column='PD_Origination',
    lgd_observed_column='LGD_Observed',        
    stage_column='Stage',
    dpd_column='DaysPastDue',
    product_type_column='ProductType',        
    effective_rate_column='EffectiveInterestRate',
    origination_date_column='OriginationDate',
    maturity_date_column='MaturityDate',
    maturity_years_column='Maturity_Years',
    macro_date_column='Date',             

    macro_variable_mapping={
        'GDP_Growth': 'GDP_Growth',
        'UNRATE':     'UNRATE',
    },

    credit_score_column='CreditScore',
    ltv_column='LTV',
    seniority_column='Seniority',
    collateral_type_column='CollateralType',
    forbearance_flag_column='ForbearanceFlag',
    watchlist_flag_column='WatchlistFlag',

    required_columns_portfolio=[
        'LoanID', 'CurrentBalance', 'PD', 'PD_Origination',
        'Stage', 'DaysPastDue', 'EffectiveInterestRate',
        'ProductType', 'OriginationDate', 'MaturityDate', 'Maturity_Years',
    ],
    required_columns_macro=['Date', 'GDP_Growth', 'UNRATE'],
    required_columns_training=[
        'LoanID', 'Time', 'Stage', 'Default', 'EAD', 'LGD_Observed',
        'GDP_Growth', 'UNRATE',
    ],
)

validation_cfg = ValidationConfig(
    oot_holdout_quarters=4, oot_min_test_defaults=30, walk_forward_windows=5,
    pd_gini_threshold_min=0.40, pd_auc_threshold_min=0.70,
    pd_psi_threshold_warning=0.10, pd_psi_threshold_fail=0.25,
    pd_binomial_test_alpha=0.05, pd_hosmer_lemeshow_bins=10,
    pd_hosmer_lemeshow_alpha=0.05, lgd_mae_threshold_max_pct=15.0,
    lgd_correlation_threshold_min=0.30, lgd_rmse_threshold_max_pct=20.0,
    lgd_downturn_bias_max_pct=5.0, lgd_downturn_under_prediction_max_pct=2.5,
    ead_correlation_threshold_min=0.40, ead_rmse_threshold_max_pct=20.0,
    ead_mpe_bias_max_pct=5.0, backtest_min_observations=50, backtest_min_defaults=10,
    var_confidence_level=0.99, var_green_zone_max_breaches=4, var_yellow_zone_max_breaches=9,
    validation_strict=True,
)

config = CreditRiskConfig(
    accounting=accounting_cfg, basel=basel_cfg,
    model=model_cfg, data=data_cfg, validation=validation_cfg,
)
print('Config built.')

In [ ]:
BASELINE_NAME = baseline_raw['Scenario Name'].iloc[0]
SEVERE_NAME   = severe_raw['Scenario Name'].iloc[0]

print(f"Detected Baseline Name: '{BASELINE_NAME}'")
print(f"Detected Severe Name:   '{SEVERE_NAME}'")


FRB_COLUMN_MAPPING = {
    'GDP_Growth': 'Real GDP growth',
    'UNRATE':     'Unemployment rate',
}

scenarios = {
    BASELINE_NAME: baseline_raw,
    SEVERE_NAME:   severe_raw,
}

SCENARIO_WEIGHTS = {
    BASELINE_NAME: 0.60,
    SEVERE_NAME:   0.40,
}

print('FRB baseline GDP range:', baseline_raw['Real GDP growth'].min(), '→', baseline_raw['Real GDP growth'].max())
print('FRB severe   GDP range:', severe_raw['Real GDP growth'].min(), '→', severe_raw['Real GDP growth'].max())

In [ ]:
tm_model = TransitionMatrixModel(config=config)

historical_tms = tm_model.estimate_from_raw_history(
    raw_history_df=training_data,
    alpha=0.001,
 
)

print(f'TMs estimated: {len(historical_tms)} (one per quarter in training_data)')
print('\nLong-run transition matrix (rows = from stage, cols = to stage):')
print('           S1       S2       S3')
for i, row_name in enumerate(['S1', 'S2', 'S3']):
    vals = tm_model.long_run_tm[i]
    print(f'  {row_name}:  {vals[0]:.4f}   {vals[1]:.4f}   {vals[2]:.4f}')

In [ ]:
z_series, rho_optimal, bin_boundaries = tm_model.compress_to_z_score()

print(f'Optimal rho:  {rho_optimal:.4f}')
print(f'Z-series:     mean={z_series.mean():.4f}  std={z_series.std():.4f}')
print(f'              min={z_series.min():.4f}   max={z_series.max():.4f}')

cc_model = CreditCycleModel(config=config)

bridge_model = cc_model.estimate_bridge_equation(
    z_series=z_series,
    macro_data=macro_history,
)

print(f'\nBridge R\u00b2: {bridge_model.rsquared:.4f}')
print('Coefficients:')
for var, coef in bridge_model.params.items():
    print(f'  {var}: {coef:.4f}')

In [ ]:
pd_model = TTCToPITAdapter(
    base_model=PrecomputedModel(column='PD', name='portfolio_PD'),
    config=config,
    pd_floor=0.0003,      
)

def bank_lgd(data: pd.DataFrame) -> np.ndarray:
    """Replace with your institution's fitted LGD model."""
    lgd = np.full(len(data), 0.40)
    if 'Seniority' in data.columns:
        lgd = np.where(data['Seniority'] == 'Senior Secured', 0.25, lgd)
    if 'CollateralType' in data.columns:
        lgd = np.where(data['CollateralType'] == 'Real Estate', lgd * 0.80, lgd)
        lgd = np.where(data['CollateralType'] == 'Vehicle',     lgd * 0.90, lgd)
    return lgd

lgd_model = DownturnLGDAdapter(
    base_model=bank_lgd,
    config=config,
    sensitivity=0.04,   
    lgd_floor=0.05,
    lgd_cap=1.0,
)

portfolio['IsRevolving'] = (portfolio['ProductType'] == 'Credit_Card')

ead_model = StressedEADAdapter(
    base_model=PrecomputedModel(column='CurrentBalance', name='EAD_balance'),
    config=config,
    stress_multiplier=0.03,
    cap_multiplier=1.5,
    revolving_only=True,
    revolving_flag_col='IsRevolving',  
)

models = {'pd': pd_model, 'lgd': lgd_model, 'ead': ead_model}
print('Models assembled.')

In [ ]:
print('FRB baseline name:', baseline_raw['Scenario Name'].iloc[0])
print('FRB severe   name:', severe_raw['Scenario Name'].iloc[0])
print()
print('FRB baseline GDP range:', baseline_raw['Real GDP growth'].min(), '→', baseline_raw['Real GDP growth'].max())
print('FRB severe   GDP range:', severe_raw['Real GDP growth'].min(), '→', severe_raw['Real GDP growth'].max())
print('FRB baseline UNRATE:   ', baseline_raw['Unemployment rate'].min(), '→', baseline_raw['Unemployment rate'].max())
print('FRB severe   UNRATE:   ', severe_raw['Unemployment rate'].min(), '→', severe_raw['Unemployment rate'].max())
print()

In [ ]:
INITIAL_CAPITAL = 150_000_000  

result = stress_test(
    portfolio=portfolio,
    scenarios=scenarios,               
    models=models,
    config=config,

    macro_history=macro_history,       
    training_data=training_data,       

    initial_capital=INITIAL_CAPITAL,
    scenario_weights=SCENARIO_WEIGHTS,
    forecast_quarters=9,

    column_mapping=FRB_COLUMN_MAPPING,

    baseline_scenario_name=BASELINE_NAME,
    severe_scenario_name=SEVERE_NAME,

    rwa_stress_multipliers={SEVERE_NAME: 1.15},

    run_model_validation=False,
    run_data_quality=True,
)

print(f'Complete: {result.run_timestamp}')

In [ ]:
print('=== ECL Summary by Scenario and Stage ===')
display(result.ecl_summary)

print(f'\nTotal Weighted ECL:  ${result.total_weighted_ecl:,.0f}')
print(f'ECL Multiplier:      {result.ecl_multiplier:.2f}x  (Severe / Baseline)'
      f' — target 1.5x–7.0x')
print(f'Bridge R²:           {result.bridge_r_squared:.4f}  — minimum 0.20 required')
print(f'Optimal Rho:         {result.rho_optimal:.4f}')
print(f'Min CET1:            {result.min_cet1_ratio:.2%}  — regulatory min 7.0%')
print(f'Passes Min Capital:  {result.passes_regulatory_minimum}')

In [ ]:
print('=== Scenario Comparison ===')
display(result.scenario_comparison)

print('\n=== Capital Trajectory ===')
display(result.capital_trajectory)

In [ ]:
plausibility = result.validate_economic_plausibility(
    expected_multiplier_min=1.5,
    expected_multiplier_max=7.0,
)
for k, v in plausibility.items():
    print(f'  {k}: {v}')

In [ ]:
result.plot('capital_adequacy', save_path='capital_adequacy.html').show()
result.plot('scenario_comparison', save_path='ecl_comparison.html').show()
result.to_excel('CCAR_stress_test.xlsx')
print('Exported to CCAR_stress_test.xlsx')